# 06 — Binary & Severe Disruption Models

The four-class baseline struggled with the rare `cancelled` and `severe_delay` classes. This notebook reformulates the problem into two operational binary tasks while preserving the same temporal split:

1. **Any disruption:** `normal` vs `delay | severe_delay | cancelled`.
2. **Severe disruption:** `severe_delay | cancelled` vs everything else.

Thresholds are tuned on 2024 validation data only. The 2025 test set remains untouched until final evaluation.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    balanced_accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, log_loss,
    precision_recall_curve
)

cwd = Path.cwd().resolve()
if (cwd / 'data').exists(): ROOT = cwd
elif (cwd.parent / 'data').exists(): ROOT = cwd.parent
else: raise FileNotFoundError(f'Cannot locate project root from {cwd}')

FEATURE_FILE = ROOT / 'data/processed/ogg_model_features_v1.csv.gz'
SCHEMA_FILE = ROOT / 'data/processed/ogg_model_features_v1_schema.json'
print('Feature file exists:', FEATURE_FILE.exists())


## 1. Load data and preserve the temporal split

In [ ]:
df = pd.read_csv(FEATURE_FILE, low_memory=False)
print('Shape:', df.shape)
print('Columns:', len(df.columns))

if 'split' not in df.columns:
    raise KeyError("Expected saved 'split' column from Notebook 04.")

train_df = df[df['split'].eq('train')].copy()
val_df = df[df['split'].eq('validation')].copy()
test_df = df[df['split'].eq('test')].copy()

display(pd.DataFrame({
    'split':['train','validation','test'],
    'rows':[len(train_df),len(val_df),len(test_df)]
}))
display(pd.crosstab(df['split'], df['disruption_class']))


## 2. Define the two operational targets

In [ ]:
for part in [train_df, val_df, test_df]:
    part['target_any_disruption'] = (part['disruption_class'] != 'normal').astype(int)
    part['target_severe'] = part['disruption_class'].isin(['severe_delay','cancelled']).astype(int)

for target in ['target_any_disruption','target_severe']:
    rows = []
    for name, part in [('train',train_df),('validation',val_df),('test',test_df)]:
        rows.append({
            'split': name,
            'positive_n': int(part[target].sum()),
            'positive_pct': 100 * part[target].mean(),
        })
    print('\n', target)
    display(pd.DataFrame(rows).round(3))


## 3. Select leakage-safe predictors and audit feature types

In [ ]:
exclude = {
    'disruption_class','target_any_disruption','target_severe','split',
    'FlightDate','ogg_sched_dt','weather_dt',
    'Cancelled','CancellationCode','Diverted','DepTime','ArrTime',
    'DepDelay','ArrDelay','DepDelayMinutes','ArrDelayMinutes',
    'CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay',
}
feature_cols = [c for c in train_df.columns if c not in exclude and not train_df[c].isna().all()]
numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train_df[c])]
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

print('Features:', len(feature_cols))
print('Numeric:', len(numeric_cols))
print('Categorical:', len(categorical_cols))
display(pd.DataFrame({
    'feature': feature_cols,
    'type': ['numeric' if c in numeric_cols else 'categorical' for c in feature_cols]
}))

for c in numeric_cols:
    if not pd.api.types.is_numeric_dtype(train_df[c]):
        raise TypeError(f'{c} was classified numeric but has dtype {train_df[c].dtype}')
print('Feature-type audit passed.')


## 4. Shared preprocessing and model definitions

In [ ]:
linear_preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), numeric_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), categorical_cols),
])

tree_preprocess = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ]), categorical_cols),
])

def make_logreg():
    return Pipeline([
        ('preprocess', linear_preprocess),
        ('model', LogisticRegression(
            C=1.0, max_iter=1000, class_weight='balanced',
            solver='lbfgs', tol=1e-4
        )),
    ])

def make_histgb():
    return Pipeline([
        ('preprocess', tree_preprocess),
        ('model', HistGradientBoostingClassifier(
            learning_rate=0.08, max_iter=250, max_leaf_nodes=31,
            min_samples_leaf=30, l2_regularization=1.0,
            class_weight='balanced', random_state=42
        )),
    ])


## 5. Evaluation helpers and validation-threshold tuning

In [ ]:
def probability_metrics(y_true, prob):
    return {
        'roc_auc': roc_auc_score(y_true, prob),
        'pr_auc': average_precision_score(y_true, prob),
        'log_loss': log_loss(y_true, np.column_stack([1-prob, prob])),
    }

def threshold_table(y_true, prob):
    thresholds = np.arange(0.05, 0.96, 0.025)
    rows = []
    for t in thresholds:
        pred = (prob >= t).astype(int)
        rows.append({
            'threshold': t,
            'precision': precision_score(y_true, pred, zero_division=0),
            'recall': recall_score(y_true, pred, zero_division=0),
            'f1': f1_score(y_true, pred, zero_division=0),
            'balanced_accuracy': balanced_accuracy_score(y_true, pred),
            'predicted_positive_pct': 100 * pred.mean(),
        })
    return pd.DataFrame(rows)

def choose_threshold(tbl, min_precision=None):
    eligible = tbl.copy()
    if min_precision is not None:
        p = eligible[eligible['precision'] >= min_precision]
        if len(p): eligible = p
    return float(eligible.sort_values(['f1','recall'], ascending=False).iloc[0]['threshold'])

def report_at_threshold(y_true, prob, threshold):
    pred = (prob >= threshold).astype(int)
    m = probability_metrics(y_true, prob)
    m.update({
        'threshold': threshold,
        'accuracy': accuracy_score(y_true, pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0),
    })
    return m, pred


## 6. Task A — Any disruption risk

In [ ]:
X_train = train_df[feature_cols]
X_val = val_df[feature_cols]
X_test = test_df[feature_cols]

y_train_any = train_df['target_any_disruption']
y_val_any = val_df['target_any_disruption']
y_test_any = test_df['target_any_disruption']

models_any = {'logistic_regression': make_logreg(), 'hist_gradient_boosting': make_histgb()}
validation_any = []
trained_any = {}

for name, model in models_any.items():
    print('Training', name)
    model.fit(X_train, y_train_any)
    prob = model.predict_proba(X_val)[:,1]
    tbl = threshold_table(y_val_any, prob)
    t = choose_threshold(tbl)
    metrics, pred = report_at_threshold(y_val_any, prob, t)
    metrics['model'] = name
    validation_any.append(metrics)
    trained_any[name] = (model, t, prob, tbl)

validation_any_df = pd.DataFrame(validation_any).sort_values('pr_auc', ascending=False)
display(validation_any_df.round(4))


In [ ]:
best_any_name = validation_any_df.iloc[0]['model']
best_any_model, best_any_t, val_prob_any, any_thr = trained_any[best_any_name]
print('Selected any-disruption model:', best_any_name)
print('Validation-selected threshold:', best_any_t)
display(any_thr.sort_values('f1', ascending=False).head(10).round(4))

test_prob_any = best_any_model.predict_proba(X_test)[:,1]
test_any_metrics, test_pred_any = report_at_threshold(y_test_any, test_prob_any, best_any_t)
display(pd.DataFrame([test_any_metrics]).round(4))
print(classification_report(y_test_any, test_pred_any, target_names=['normal','disrupted'], zero_division=0))
display(pd.DataFrame(confusion_matrix(y_test_any, test_pred_any), index=['true_normal','true_disrupted'], columns=['pred_normal','pred_disrupted']))


## 7. Task B — Severe operational disruption

For the severe target, PR-AUC is especially important because positives are rare. We tune thresholds on validation F1, but also inspect alternatives with minimum precision constraints.

In [ ]:
y_train_severe = train_df['target_severe']
y_val_severe = val_df['target_severe']
y_test_severe = test_df['target_severe']

models_severe = {'logistic_regression': make_logreg(), 'hist_gradient_boosting': make_histgb()}
validation_severe = []
trained_severe = {}

for name, model in models_severe.items():
    print('Training', name)
    model.fit(X_train, y_train_severe)
    prob = model.predict_proba(X_val)[:,1]
    tbl = threshold_table(y_val_severe, prob)
    t = choose_threshold(tbl)
    metrics, pred = report_at_threshold(y_val_severe, prob, t)
    metrics['model'] = name
    validation_severe.append(metrics)
    trained_severe[name] = (model, t, prob, tbl)

validation_severe_df = pd.DataFrame(validation_severe).sort_values('pr_auc', ascending=False)
display(validation_severe_df.round(4))


In [ ]:
best_severe_name = validation_severe_df.iloc[0]['model']
best_severe_model, best_severe_t, val_prob_severe, severe_thr = trained_severe[best_severe_name]
print('Selected severe-disruption model:', best_severe_name)
print('Validation-selected threshold:', best_severe_t)
display(severe_thr.sort_values('f1', ascending=False).head(12).round(4))

print('Best thresholds with precision >= 0.10, if available:')
display(severe_thr[severe_thr['precision'] >= 0.10].sort_values('recall', ascending=False).head(10).round(4))

test_prob_severe = best_severe_model.predict_proba(X_test)[:,1]
test_severe_metrics, test_pred_severe = report_at_threshold(y_test_severe, test_prob_severe, best_severe_t)
display(pd.DataFrame([test_severe_metrics]).round(4))
print(classification_report(y_test_severe, test_pred_severe, target_names=['not_severe','severe'], zero_division=0))
display(pd.DataFrame(confusion_matrix(y_test_severe, test_pred_severe), index=['true_not_severe','true_severe'], columns=['pred_not_severe','pred_severe']))


## 8. Precision–recall curves on validation data

In [ ]:
for title, y_true, prob in [
    ('Any disruption — validation', y_val_any, val_prob_any),
    ('Severe disruption — validation', y_val_severe, val_prob_severe),
]:
    precision, recall, _ = precision_recall_curve(y_true, prob)
    plt.figure(figsize=(7,5))
    plt.plot(recall, precision)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(title)
    plt.grid(alpha=0.25)
    plt.show()


## 9. Interpretation checkpoint

Record the following before moving to recovery-time modeling:

- validation and test PR-AUC for both tasks;
- the validation-selected thresholds;
- severe-disruption recall and precision on 2025;
- whether HistGradientBoosting materially improves over logistic regression;
- whether the severe target is predictable enough from current weather/schedule features to support user-facing risk alerts.

Next, we can add strictly historical airline/route disruption-rate features and then build the separate weather-event recovery-time model.